## Importing

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer # Used for Stemming
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download("stopwords")

In [ ]:
print(stopwords.words('english'))

## Data Processing

In [ ]:
# Naming the columns and Reading the dataset

column_names = ['target', 'id', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv("/content/training.1600000.processed.noemoticon.csv",names = column_names, encoding = 'ISO-8859-1')

In [ ]:
# Check the number of rows and columns

twitter_data.shape

In [ ]:
# Printing the first 5 rows of the dataframe

twitter_data.head()

In [ ]:
# Checking for Null Values

twitter_data.isnull().sum()

In [ ]:
# Checking for duplicates

twitter_data.duplicated().sum()

In [ ]:
# Checking the distribution of target

twitter_data['target'].value_counts()

## Converting the target "4" to "1"

In [ ]:
twitter_data.replace({'target':{4:1}}, inplace=True)

In [ ]:
# All the values of "4" converted to "1"

twitter_data['target'].value_counts()

- 0 -- Negative Tweet

- 1 -- Positive Tweet

## **Stemming**

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):

  stemmed_content = re.sub('[^a-zA-Z]',' ', content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)

  return stemmed_content

In [ ]:
!pip install swifter

In [ ]:
import swifter
twitter_data['stemmed_content'] = twitter_data['text'].swifter.apply(stemming)

# twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)

In [ ]:
twitter_data.head()

In [ ]:
print(twitter_data['stemmed_content'])

In [ ]:
print(twitter_data['target'])

In [ ]:
# Separating the data and label

X = twitter_data['stemmed_content'].values
Y = twitter_data['target'].values

# Data Split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify = Y, random_state=42)

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

In [ ]:
# Converting textual data to numerical data

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# Training Machine Learning Models

In [ ]:
model_lr = LogisticRegression(max_iter=1000)

In [ ]:
model_lr.fit(X_train, Y_train)

In [ ]:
## Accuracy on the training data

X_train_prediction = model_lr.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [ ]:
print("Accuracy on Training Data", training_data_accuracy)

In [ ]:
## Accuracy on the test data

X_test_prediction = model_lr.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

In [ ]:
print("Accuracy on Test Data", test_data_accuracy)

In [ ]:
import pickle


In [ ]:
filename = 'trained_model.sav'

pickle.dump(model_lr, open(filename, 'wb'))

## Using the saved model for future prediction

In [ ]:
# Loading the saved model
loaded_model =pickle.load(open('trained_model.sav', 'rb'))

# 'trained_model.sav' or copy path of the file

In [ ]:
X_new = X_test[200]
print(Y_test[200])

prediction = model_lr.predict(X_new)
print(prediction)

if (prediction[0] == 0):
  print("Negative Tweet")
else:
  print("Positive Tweet")